# E10 Adult Census：折内目标编码与分组表现

> Issue #9 / Epic #6 / 路线图 E10

## 你将学到
1. 目标编码必须在折内拟合，否则泄漏
2. 不同人群（性别等）表现差异需单独评估
3. 高基数类别处理

## 数据
- 本地 `data/adult.csv` 或 OpenML `adult`


In [ ]:
QUICK = True
RANDOM_STATE = 42

import warnings
warnings.filterwarnings('ignore')

from wanglaoshi import VERSIONS as V
from wanglaoshi import JupyterEnv as JE

JE.no_warning()
V.show_env_info()


## 1. 加载 Adult


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

path = Path('../../data/adult.csv')
if path.exists():
    df = pd.read_csv(path)
    print('local', df.shape)
else:
    from sklearn.datasets import fetch_openml
    raw = fetch_openml('adult', version=2, as_frame=True, parser='auto')
    df = raw.frame.copy()
    print('OpenML', df.shape)

# 目标列统一
target_col = None
for c in ['income', 'class', 'target', '>50K']:
    if c in df.columns:
        target_col = c
        break
assert target_col, list(df.columns)
print('target', target_col, df[target_col].value_counts())
df.head()


## 2. EDA


In [ ]:
from wanglaoshi import DataAnalyzer

eda_df = df.sample(min(len(df), 5000), random_state=RANDOM_STATE) if QUICK else df
analyzer = DataAnalyzer(eda_df)
display(analyzer.missing_value_analysis().head(15))
display(analyzer.basic_statistics().head(15))


## 3. 折内目标编码 vs 错误的全量编码


In [ ]:
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import LabelEncoder

work = df.copy()
# 二值化目标
y_raw = work[target_col].astype(str)
y = (y_raw.str.contains('>50K')).astype(int) if y_raw.str.contains('>50K').any() else LabelEncoder().fit_transform(y_raw)
work = work.drop(columns=[target_col])

cat_cols = work.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
num_cols = work.select_dtypes(include=[np.number]).columns.tolist()
print('cat', cat_cols[:8], 'num', num_cols[:8])

Xtr, Xte, ytr, yte = train_test_split(
    work, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

def target_encode_inplace(train, valid, y_train, cols):
    train = train.copy(); valid = valid.copy()
    for c in cols:
        means = y_train.groupby(train[c].astype(str)).mean()
        global_mean = y_train.mean()
        train[c + '_te'] = train[c].astype(str).map(means).fillna(global_mean)
        valid[c + '_te'] = valid[c].astype(str).map(means).fillna(global_mean)
    return train, valid

# ✅ 只用训练集统计做 TE（仍建议进一步用 KFold；这里先演示 train-only map）
Xtr_te, Xte_te = target_encode_inplace(Xtr, Xte, pd.Series(ytr, index=Xtr.index), cat_cols[:6])

# ❌ 错误：用全量（含验证）统计 —— 仅作对比，勿上线
# full_means = pd.Series(y, index=work.index).groupby(work[c]).mean()

feat = num_cols + [c + '_te' for c in cat_cols[:6]]
Xtr_f = Xtr_te[feat].apply(pd.to_numeric, errors='coerce').fillna(0)
Xte_f = Xte_te[feat].apply(pd.to_numeric, errors='coerce').fillna(0)
print(Xtr_f.shape)


## 4. QuickBaseline


In [ ]:
from wanglaoshi import CompetitionTools as CT
from sklearn.metrics import classification_report, roc_auc_score

baseline = CT.QuickBaseline()
results = baseline.run_all_models(Xtr_f, pd.Series(ytr), Xte_f, pd.Series(yte), task_type='classification')
comparison = baseline.compare_models(results, top_n=5)
display(comparison)
best = comparison.iloc[0]['Model']
pred = results[best]['predictions']
print(classification_report(yte, pred, zero_division=0))


## 5. 分组表现（如性别）


In [ ]:
group_col = 'sex' if 'sex' in Xte.columns else ('Sex' if 'Sex' in Xte.columns else None)
if group_col:
    tmp = pd.DataFrame({'y': yte, 'pred': pred, 'g': Xte[group_col].astype(str).to_numpy()})
    print(tmp.groupby('g').apply(lambda d: (d['y'] == d['pred']).mean()))
else:
    print('无 sex 列，跳过分组评估；可用 race/gender 等列自行替换')


## 6. 模拟提交


In [ ]:
from pathlib import Path
from wanglaoshi import CompetitionTools as CT

out = Path('outputs/10_adult_census'); out.mkdir(parents=True, exist_ok=True)
sample = pd.DataFrame({'Id': np.arange(len(pred)), 'income': np.asarray(yte)})
sample.to_csv(out / 'sample_submission.csv', index=False)
gen = CT.SubmissionGenerator()
sub = gen.create_submission(
    pred, str(out / 'sample_submission.csv'), str(out / 'submission.csv'), prediction_type='label'
)
print(gen.validate_submission(str(out / 'submission.csv'), str(out / 'sample_submission.csv')))


## 7. 检查清单
- [x] train-only 目标编码
- [x] 分组表现评估入口
- [ ] 真正的 KFold 折内 TE
- [ ] 公平性指标（TPR 差距等）
